# Find daily meal plan

In [1]:
import os,sys
print(os.getcwd())
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print("Project root directory:", project_root)
sys.path.append(project_root)

/home/zhaolei/meal-rec-ai/preprocess
Project root directory: /home/zhaolei/meal-rec-ai


In [2]:
import pandas as pd
import warnings
from utils.data import root_path
import os
warnings.filterwarnings('ignore')
data_path=f'{root_path}daily_nutrition.csv'
data = pd.read_csv(data_path)
data['daily_food_id'] = data['user_id'].astype(str) + data['years'].astype(str) +  data['day'].astype(int).astype(str)
data = data.drop(columns=['years', 'day'])
pd.set_option('display.max_colwidth', None) 
data.columns

Index(['user_id', 'age_group', 'gender', 'grams', 'calorie', 'protein', 'carb',
       'sugar', 'fiber', 'saturated_fat', 'cholesterol', 'folic_acid',
       'vitamin_b12', 'vitamin_c', 'vitamin_d', 'calcium', 'phosphorus',
       'potassium', 'iron', 'sodium', 'age', 'user_low_phosphorus',
       'user_low_carb', 'weight', 'height', 'under_weight', 'over_weight',
       'user_low_calorie', 'user_high_calorie', 'user_low_sodium',
       'user_high_potassium', 'blood_pressure', 'user_low_saturated_fat',
       'user_low_cholesterol', 'low_density_lipoprotein',
       'blood_urea_nitrogen', 'user_low_protein', 'user_high_protein',
       'opioid_misuse', 'diabetes', 'user_low_sugar', 'user_high_fiber',
       'anemia', 'user_high_vitamin_b12', 'user_high_folate_acid',
       'user_high_iron', 'user_high_vitamin_c', 'user_high_calcium',
       'user_high_vitamin_d', 'osteoporosis', 'level', 'b_calorie', 'b_carb',
       'b_fiber', 'b_protein', 'b_saturated_fat', 'b_sugar', 'b_cholesterol'

# Mine Gold level daily meal plan
Indicates a meal plan that exists in the NHANES dataset and is consumed by users who are within their personal target range.

In [3]:
# 定义函数应用 evaluate 类
from utils.health_evaluate import evaluate_nutrition
from utils.target_calculation import get_nutrition_target_range_by_situation,medical_cols
data=evaluate_nutrition(data)
current_match=data[data['match']==True]
current_match=current_match[current_match['age']>18]
current_match['target'] = current_match.apply(get_nutrition_target_range_by_situation, axis=1)
print(f'match daily food:{len(current_match)}, \
    user: {len(current_match.drop_duplicates(subset=["user_id"], keep="first"))}')

Project root directory: /home/zhaolei/meal-rec-ai
match daily food:148,     user: 125


In [4]:
current_match.to_csv(f'{root_path}daily_food_with_nutrition_target_gold.csv',index=False)

## Golden level: 125 users ate 148 daily meal plan within defined target. 

# Mine Silver level daily meal plan
Indicates a meal plan that exists in the NHANES dataset but is consumed by users who are not within their personal target range.

In [4]:
def get_food_data(data):
    food= data.copy()
    food= food.drop(columns=['macro_health_score','micro_health_score','gender',
                             'age', 'weight', 'height', 'level','user_id']+medical_cols)
    return food
def get_user_data(data):
    user = data[data['match']==False]
    user= user.drop(columns=['grams', 'calorie', 'protein', 'carb', 'sugar', 'fiber',
           'saturated_fat', 'cholesterol', 'folic_acid', 'vitamin_b12',
           'vitamin_c', 'vitamin_d', 'calcium', 'phosphorus', 'potassium', 'iron',
           'sodium', 'macro_health_score','micro_health_score', 'total_positive_score', 'total_negative_score',
           'b_calorie', 'b_carb', 'b_protein', 'b_saturated_fat', 'b_sugar',
           'b_cholesterol','b_fiber',  'b_sodium', 'b_phosphorus',
           'b_potassium', 'b_iron', 'b_calcium', 'b_folic_acid', 'b_vitamin_c', 'b_vitamin_d',
           'b_vitamin_b12','daily_food_id'])
    user=user[user['age']>18]
    user = user.drop_duplicates(subset=['user_id'], keep='first')
    # Apply the function and assign results to columns
    user['target'] = user.apply(get_nutrition_target_range_by_situation, axis=1)
    return user

food=get_food_data(data)
user=get_user_data(data)
print(f'food: {len(food)}, user: {len(user)}')
user.columns

food: 33348, user: 23068


Index(['user_id', 'age_group', 'gender', 'age', 'user_low_phosphorus',
       'user_low_carb', 'weight', 'height', 'under_weight', 'over_weight',
       'user_low_calorie', 'user_high_calorie', 'user_low_sodium',
       'user_high_potassium', 'blood_pressure', 'user_low_saturated_fat',
       'user_low_cholesterol', 'low_density_lipoprotein',
       'blood_urea_nitrogen', 'user_low_protein', 'user_high_protein',
       'opioid_misuse', 'diabetes', 'user_low_sugar', 'user_high_fiber',
       'anemia', 'user_high_vitamin_b12', 'user_high_folate_acid',
       'user_high_iron', 'user_high_vitamin_c', 'user_high_calcium',
       'user_high_vitamin_d', 'osteoporosis', 'level', 'match', 'target'],
      dtype='object')

# Priority Setting by User

essentional_nutritions = ['calorie', 'protein', 'carb', 'saturated_fat', 'cholesterol']

nutrition_by_situation = {

    'health': [],
    'under_weight': [],
    'over_weight': ['sugar'],
    'blood_pressure': ['sodium', 'potassium'],
    'low_density_lipoprotein': ['fiber'],
    'blood_urea_nitrogen': [],
    'diabetes': ['sugar', 'sodium'],
    'anemia': ['iron', 'vitamin_c', 'folic_acid', 'vitamin_b12'],
    'osteoporosis': ['calcium', 'vitamin_d', 'vitamin_c'],
    'opioid_misuse': ['fiber', 'sugar', 'sodium'],
    'user_low_phosphorus': ['phosphorus'],
    'user_low_saturated_fat': [],
    'user_low_carb': [],
    'user_low_calorie': [],
    'user_low_cholesterol': [],
    'user_low_sugar': ['sugar'],
    'user_high_calorie': [],
    'user_low_protein': [],
    'user_low_sodium': ['sodium'],
    'user_high_potassium': ['potassium'],
    'user_high_protein': [],
    'user_high_fiber': ['fiber'],
    'user_high_folate_acid':['folic_acid'],
    'user_high_iron':['iron'],
    'user_high_vitamin_b12': ['vitamin_b12'],
    'user_high_calcium': ['calcium'],
    'user_high_vitamin_c': ['vitamin_c'],
    'user_high_vitamin_d': ['vitamin_d']
}


## Find in-target daily food for each user (GPU)

In [5]:
from utils.find_silver import search_match_food_torch
# 使用示例
user2=user.drop(columns=['match','age_group'])
found, not_found = search_match_food_torch(user2, food)
print(f'found: {len(found)}')
print(f'not found user: {len(not_found)}')

Processing Users: 100%|██████████| 23068/23068 [00:03<00:00, 6169.94user/s]

found: 11774
not found user: 11294


### Silver level: expand: 11774 daily meal plan serve for new users without experience history 
### not found in-target user: 11294

In [7]:
found=evaluate_nutrition(found)
found['target'] = found.apply(get_nutrition_target_range_by_situation, axis=1)
found

,user_id,gender,age,user_low_phosphorus,user_low_carb,weight,height,under_weight,over_weight,user_low_calorie,...,b_folic_acid,b_vitamin_c,b_vitamin_d,b_vitamin_b12,match,total_positive_score,total_negative_score,daily_food_id,macro_health_score,micro_health_score
0,21010,2.0,52,0,0,67.9,163.2,0,1,1,...,True,True,True,True,True,2,0,7118011121,7,8
1,21012,1.0,63,0,0,59.2,173.8,0,0,0,...,True,True,True,False,True,5,7,210963042,6,8
2,21017,2.0,37,0,0,45.1,152.7,0,0,0,...,True,True,True,True,True,3,6,210183041,6,7
3,21018,2.0,33,0,0,44.9,164.6,1,0,0,...,True,True,True,False,True,7,2,254713041,6,8
4,21019,2.0,50,0,0,88.1,152.2,0,1,1,...,True,True,True,True,True,2,6,281583042,7,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11769,124723,1.0,27,0,0,54.8,156.4,0,0,0,...,True,True,True,True,True,7,2,210853041,6,8
11770,124771,2.0,25,0,0,62.7,167.7,0,0,0,...,True,True,True,True,True,7,2,210853041,6,8
11771,124775,2.0,54,0,0,61.3,156.5,0,0,0,...,True,True,True,True,True,11,4,210523041,6,8
11772,124792,1.0,32,0,0,61.9,168.7,0,0,0,...,True,True,True,True,True,7,2,210853041,6,8


In [8]:
found.to_csv(f'{root_path}daily_food_with_nutrition_target_silver.csv',index=False)

In [6]:
not_found.to_csv(f'{root_path}not_found_match_food_user.csv')
not_found['target'] = not_found.apply(get_nutrition_target_range_by_situation, axis=1)
not_found_target=not_found['target'].drop_duplicates()
print(f'user count: {len(not_found)}, missed targets: {len(not_found_target)}')

user count: 11294, missed targets: 4019


# Mine Bronze level daily meal plan
Indicates a meal plan that does not exist in the NHANES dataset and is generated using a genetic algorithm from the NHANES food library based on the user's target range.

### Run Batch
suggest to run below code in command-line batch background mode. It costs  1 day, 19:27:16.853742 for first time mining.

- Intel(R) Xeon(R) Gold 5218B CPU @ 2.30GHz 16 cores x 2
- 128GB memory
- RTX A6000 GPU

- change population size and generations to handle not found targets

```bash
python find_bronze.py --data='data/nhanes/data' -p=3000 -d=0 -g=100 -c=0.2 -m=0.9
python find_bronze.py --data='data/nhanes/data' -p=4000 -d=0 -g=100 -c=0.2 -m=0.9
python find_bronze.py --data='data/nhanes/data' -p=5000 -d=0 -g=100 -c=0.2 -m=0.9
python find_bronze.py --data='data/nhanes/data' -p=6000 -d=0 -g=100 -c=0.2 -m=0.9
...
```

| # | Parameter | Duration | Found | Not Found |
|-----------------|---------|------------------|--------------|-----------|
| 1 | `-p=3000 -g=100 -d=0 -c=0.2 -m=0.9` | 1 day, 19:27:16 | 2836 | 1183 |
| 2 | `-p=4000 -g=100 -d=0 -c=0.2 -m=0.9` | 1 day, 1:57:48 | 3457 | 562 |
| 3 | `-p=5000 -g=100 -d=0 -c=0.2 -m=0.9` | 14:19:08 | 3710 | 309 |
| 4 | `-p=5000 -g=100 -d=0 -c=0.2 -m=0.9` | 8:49:23 | 3804 | 215 |
| 5 | `-p=5000 -g=100 -d=0 -c=0.2 -m=0.9` | 8:00:29 | 3856 | 163 |
| 6 | `-p=5000 -g=100 -d=0 -c=0.2 -m=0.9` | 4:13:41 | 3904 | 115 |
| 7 | `-p=5000 -g=100 -d=0 -c=0.2 -m=0.9` | 3:35:20 | 3922 | 97 |
| 8 | `-p=5000 -g=100 -d=0 -c=0.2 -m=0.9` | 3:04:09 | 3934 | 85 |
| 9 | `-p=5000 -g=100 -d=0 -c=0.2 -m=0.9` | 2:42:17 | 3943 | 76 |
| 10 | `-p=5000 -g=100 -d=0 -c=0.2 -m=0.9` | 2:35:25 | 3950 | 69 |
| 11 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 3:13:51 | 3959  | 60  |
| 12 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 2:20:54 | 3963  | 56  |
| 13 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 2:40:50 | 3967  | 52  |
| 14 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 2:03:05 | 3972  | 47  |
| 15 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 1:51:05 | 3973  | 46  |
| 16 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 2:20:49 | 3976  | 43  |
| 17 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 1:53:18 | 3977  | 42  |
| 18 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 1:40:29 | 3978  | 41  |
| 19 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 1:59:44 | 3980  | 39  |
| 20 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 2:00:27 | 3982  | 37  |
| 21 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 1:26:05 | 3985  | 34  |
| 22 | `-p=6000 -g=100 -d=0 -c=0.2 -m=0.9` | 1:19:42 | 3985  | 34  |

## Save Result

In [7]:
def save_to_bronze_data(bronze_df,user):
    bronze_df=bronze_df.drop_duplicates()
    bronze_df['daily_food_id'] = bronze_df['user_id'].astype(str) + bronze_df['years'].astype(str) +  bronze_df['day'].astype(int).astype(str)
    daily_bronze_df = bronze_df.groupby(['daily_food_id','target','user_id']).agg({
        'grams': 'sum',
        'calorie': 'sum',
        'protein': 'sum',
        'carb': 'sum',
        'sugar': 'sum',
        'fiber': 'sum',
        'saturated_fat': 'sum',
        'cholesterol': 'sum',
        'folic_acid': 'sum',
        'vitamin_b12': 'sum',
        'vitamin_c': 'sum',
        'vitamin_d': 'sum',
        'calcium': 'sum',
        'phosphorus': 'sum',
        'potassium': 'sum',
        'iron': 'sum',
        'sodium': 'sum'
        }).reset_index()
    daily_bronze_df = pd.merge(daily_bronze_df, user[['user_id','age_group', 'gender', 'age',
       'user_low_phosphorus', 'user_low_carb', 'weight', 'height',
       'under_weight', 'over_weight', 'user_low_calorie', 'user_high_calorie',
       'user_low_sodium', 'blood_pressure', 'user_high_potassium',
       'user_low_saturated_fat', 'user_low_cholesterol',
       'low_density_lipoprotein', 'blood_urea_nitrogen', 'user_low_protein',
       'user_high_protein', 'opioid_misuse', 'user_low_sugar',
       'user_high_fiber', 'diabetes', 'user_high_folate_acid',
       'user_high_iron', 'user_high_vitamin_b12', 'anemia', 'osteoporosis',
       'user_high_calcium', 'user_high_vitamin_c', 'user_high_vitamin_d',
       'level', 'match']], on=['user_id'], how='left')
    daily_bronze_df.to_csv(f'{root_path}daily_food_with_nutrition_target_bronze.csv', index=False)
    bronze_df[['daily_food_id','food_id','user_id','eating_type','grams','day','years']].to_csv(f'{root_path}food_user_bronze.csv', index=False)
    print('save completed!')
    return bronze_df,daily_bronze_df

In [9]:
bronze_df = pd.read_csv(f'{root_path}_daily_food_with_nutrition_target_bronze.csv')
bronze_df,daily_bronze_df=save_to_bronze_data(bronze_df,user)

save completed!
